# 03 - MSDD-Net Training

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/AfefGuesmi/msdd-arab-youth-space-hackathon/blob/main/MSDD-Net/notebooks/03_MSDDNet_Training.ipynb)

Trains MSDD-Net (Swin-UNet V2) on MARIDA. Choose a variant and run.

Runtime: about 4 to 10 hours per variant on a Colab T4 GPU. Colab Pro or a Kaggle P100 is strongly recommended.

## 1. Setup and GPU check

In [ ]:
import os
if not os.path.isdir('msdd-arab-youth-space-hackathon'):
    !git clone -b msddnet https://github.com/AfefGuesmi/msdd-arab-youth-space-hackathon.git

%cd msdd-arab-youth-space-hackathon/MSDD-Net

!pip install -q torch torchvision --index-url https://download.pytorch.org/whl/cu121
!pip install -q -r requirements.txt

import torch
assert torch.cuda.is_available(), 'Enable GPU in Runtime -> Change runtime type.'
print('GPU:', torch.cuda.get_device_name(0))

## 2. Download the ImageNet Swin-V2 checkpoint (once)

In [ ]:
from training_utils import ensure_pretrained_checkpoint
p = ensure_pretrained_checkpoint()
print('Pretrained checkpoint:', p)

## 3. Choose a variant and train

Available variants: `baseline`, `+dilated`, `+dilated+attention`, `full`, `only_dilated`, `only_attention`, `only_residual`.

Optional extra input channels: pass `--use_spectral_indices True` and/or `--use_texture_features True`.

In [ ]:
VARIANT = 'baseline'   # change to 'full', 'only_residual', etc.
EXTRA   = ''           # e.g. '--use_spectral_indices True --use_texture_features True'

In [ ]:
!python train_marida.py \
  --model mssd \
  --variant $VARIANT \
  --data_path data \
  $EXTRA

## 4. Plot training history

In [ ]:
import json, os
import matplotlib.pyplot as plt

history = f'trained_models/{VARIANT}/history.json'
assert os.path.isfile(history), f'No history file at {history}. Training did not complete.'

with open(history) as f:
    h = json.load(f)

ep   = [x['epoch'] for x in h]
tr   = [x['train_loss'] for x in h]
va   = [x['val_loss'] for x in h]
miou = [x['val_miou'] for x in h]
mf1  = [x['val_mf1'] for x in h]
md   = [x['val_md_iou'] for x in h]

fig, axes = plt.subplots(1, 3, figsize=(16, 4.5))
axes[0].plot(ep, tr, label='train')
axes[0].plot(ep, va, label='val')
axes[0].set_title('Loss'); axes[0].set_xlabel('epoch'); axes[0].legend()
axes[1].plot(ep, miou, label='mIoU')
axes[1].plot(ep, mf1,  label='mF1')
axes[1].set_title('Validation scores'); axes[1].set_xlabel('epoch'); axes[1].legend()
axes[2].plot(ep, md, color='red', label='Marine Debris IoU')
axes[2].set_title('Marine Debris IoU (target class)'); axes[2].set_xlabel('epoch'); axes[2].legend()
plt.tight_layout(); plt.show()

## 5. Save the checkpoint to Google Drive (optional)

Colab VMs are ephemeral. Copy the trained checkpoint to Drive so it survives a disconnect.

In [ ]:
SAVE_TO_DRIVE = True
if SAVE_TO_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')
    import shutil, os
    dst = f'/content/drive/MyDrive/MSDD_checkpoints/{VARIANT}'
    os.makedirs(dst, exist_ok=True)
    for f in os.listdir(f'trained_models/{VARIANT}'):
        if f.endswith(('.pth', '.json')):
            shutil.copy2(f'trained_models/{VARIANT}/{f}', dst)
    print('Saved to', dst)

## Next steps

Run **04_Evaluation_and_Figures.ipynb** to compute per-class metrics and generate the report figures.